# 📄 Notebook 1: PDF Text Extraction

This notebook demonstrates how to extract text from PDF files using:
- **PyMuPDF (fitz)** for native text extraction
- **Tesseract OCR** as fallback for scanned documents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

## 1. Install Dependencies

In [ ]:
!pip install -q PyMuPDF pdfplumber pytesseract Pillow
!apt-get install -q tesseract-ocr > /dev/null 2>&1

# Clone the repo (or upload files manually)
!git clone https://github.com/YOUR_USERNAME/ontology-builder.git 2>/dev/null || true
import sys
sys.path.insert(0, 'ontology-builder')

## 2. Upload Your PDFs

Upload PDF files using the Colab file uploader, or mount Google Drive.

In [ ]:
import os

# Option A: Upload files directly
from google.colab import files
uploaded = files.upload()

# Save uploaded files to a directory
os.makedirs('pdfs', exist_ok=True)
for filename, content in uploaded.items():
    with open(f'pdfs/{filename}', 'wb') as f:
        f.write(content)
    print(f'Saved: pdfs/{filename}')

In [ ]:
# Option B: Mount Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# PDF_DIR = '/content/drive/MyDrive/your-pdf-folder/'

## 3. Extract Text with PyMuPDF (Native)

In [ ]:
import fitz  # PyMuPDF

def extract_text_pymupdf(pdf_path):
    """Extract text from a PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    pages = []
    for page_num in range(doc.page_count):
        page = doc[page_num]
        text = page.get_text('text')
        pages.append({
            'page': page_num + 1,
            'text': text,
            'char_count': len(text),
            'method': 'native'
        })
    doc.close()
    return pages

# Test on first uploaded PDF
pdf_files = [f for f in os.listdir('pdfs') if f.endswith('.pdf')]
if pdf_files:
    pages = extract_text_pymupdf(f'pdfs/{pdf_files[0]}')
    print(f'Extracted {len(pages)} pages from {pdf_files[0]}')
    print(f'\n--- Page 1 Preview (first 500 chars) ---')
    print(pages[0]['text'][:500])
else:
    print('No PDF files found. Please upload PDFs first.')

## 4. OCR Fallback for Scanned PDFs

In [ ]:
import pytesseract
from PIL import Image
import io

def ocr_page(pdf_path, page_num, dpi=300):
    """OCR a single page by rendering to image first."""
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    pix = page.get_pixmap(dpi=dpi)
    img = Image.open(io.BytesIO(pix.tobytes('png')))
    text = pytesseract.image_to_string(img, lang='eng')
    doc.close()
    return text

def extract_with_ocr_fallback(pdf_path, min_chars=50):
    """Extract text with OCR fallback for pages with little native text."""
    doc = fitz.open(pdf_path)
    pages = []
    ocr_count = 0
    
    for page_num in range(doc.page_count):
        page = doc[page_num]
        text = page.get_text('text')
        method = 'native'
        
        if len(text.strip()) < min_chars:
            text = ocr_page(pdf_path, page_num)
            method = 'ocr'
            ocr_count += 1
        
        pages.append({
            'page': page_num + 1,
            'text': text,
            'char_count': len(text),
            'method': method
        })
    
    doc.close()
    print(f'Pages: {len(pages)} total, {ocr_count} required OCR')
    return pages

# Test
if pdf_files:
    pages = extract_with_ocr_fallback(f'pdfs/{pdf_files[0]}')
    for p in pages[:3]:
        print(f"  Page {p['page']}: {p['char_count']} chars ({p['method']})")

## 5. Using the PDFExtractor Class

The repo's `PDFExtractor` wraps all of the above into a clean interface.

In [ ]:
from src.pdf_extractor import PDFExtractor

extractor = PDFExtractor(ocr_enabled=True)

# Extract all PDFs from directory
documents = extractor.extract_from_directory('pdfs/')

for doc in documents:
    print(f'\n{doc.filename}')
    print(f'  Pages: {doc.num_pages}')
    print(f'  Total chars: {len(doc.full_text)}')
    print(f'  Metadata: {doc.metadata}')
    print(f'  Preview: {doc.full_text[:200]}...')

## 6. Save Extracted Text

Save the extracted text for use in the next notebooks.

In [ ]:
import json

os.makedirs('output', exist_ok=True)

# Save as JSON for downstream use
extracted_data = []
for doc in documents:
    extracted_data.append({
        'filename': doc.filename,
        'num_pages': doc.num_pages,
        'metadata': doc.metadata,
        'full_text': doc.full_text,
        'pages': [
            {'page': p.page_number, 'text': p.text, 'method': p.method}
            for p in doc.pages
        ]
    })

with open('output/extracted_texts.json', 'w') as f:
    json.dump(extracted_data, f, indent=2)

print(f'Saved {len(extracted_data)} documents to output/extracted_texts.json')

---
**Next:** [Notebook 2 — Concept Extraction](02_concept_extraction.ipynb)